## Federated Analysis

Here we take the weights from each of the models, aggregate them and then use these with each of the studies


In [ ]:
import numpy as np

def aggregate_weights_median(weights_dict):
    """
    Aggregate weights layer-wise across all studies using the median.
    """
    studies = list(weights_dict.keys())
    n_layers = len([w for w in weights_dict[studies[0]] if w is not None])

    aggregated_weights = []
    for layer_idx in range(n_layers):
        # Collect all weights for this layer across studies
        layer_weights = [weights_dict[study][layer_idx] for study in studies if weights_dict[study] is not None]

        # Stack and take median across the first axis (studies)
        stacked = np.stack(layer_weights, axis=0)
        median_weight = np.median(stacked, axis=0)
        aggregated_weights.append(median_weight)

    return aggregated_weights


In [ ]:
def extract_model_weights_per_study(mse_summaries):
    weights_dict = {}

    for study in mse_summaries.keys():
        model_path = f"models/{study}_model.keras"

        try:
            model = load_model(model_path)
            weights = model.get_weights()
            weights_dict[study] = weights
        except Exception as e:
            print(f"Could not load model for {study}: {e}")
            weights_dict[study] = None

    return weights_dict


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.models import model_from_json
import os

def build_model_from_file(model_path):
    """
    Load a Keras model structure from an existing model file (without weights).
    """
    model = load_model(model_path)
    model_json = model.to_json()
    new_model = model_from_json(model_json)
    return new_model


In [ ]:
def build_federated_model(aggregated_weights, reference_model_path):
    """
    Build a model with aggregated weights.
    """
    model = build_model_from_file(reference_model_path)
    model.set_weights(aggregated_weights)
    return model


In [ ]:
from sklearn.metrics import mean_squared_error
import pandas as pd

def evaluate_federated_model(model, X_test, y_true, antibody_labels):
    """
    Evaluate reconstructed output from federated model.
    """
    reconstructed = model.predict(X_test)
    mse_per_ab = {}

    for i, ab in enumerate(antibody_labels):
        mse = mean_squared_error(y_true[:, i], reconstructed[:, i])
        mse_per_ab[ab] = mse

    mse_median = np.median(list(mse_per_ab.values()))
    mse_iqr = np.subtract(*np.percentile(list(mse_per_ab.values()), [75, 25]))

    print(f"Federated Model MSE (median): {mse_median:.4f}")
    print(f"IQR: {mse_iqr:.4f}")
    print("Per-Autoantibody MSE:")
    print(pd.Series(mse_per_ab).sort_values())

    return {
        "mse_median": mse_median,
        "mse_iqr": mse_iqr,
        "per_antibody_mse": mse_per_ab,
        "reconstructed": reconstructed
    }


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

def evaluate_federated_model(federated_model, mse_summaries, study):
    summary = mse_summaries[study]
    
    X_test = summary["X_test"]
    antibodies = summary["antibody_labels"]
    age_groups = summary["Age_Group"]
    sex = summary["Sex"]

    # Reshape for CNN
    X_test_reshaped = X_test.reshape((-1, X_test.shape[1], 1, 1))
    
    # Predict
    reconstructed = federated_model.predict(X_test_reshaped)

    # Flatten CNN output
    reconstructed_flat = reconstructed.reshape(X_test.shape)

    # Overall MSE
    mse = mean_squared_error(X_test, reconstructed_flat)
    
    # Per-antibody MSE
    per_ab_mse = {
        ab: mean_squared_error(X_test[:, i], reconstructed_flat[:, i])
        for i, ab in enumerate(antibodies)
    }

    # Median of per-antibody MSEs
    mse_median = np.median(list(per_ab_mse.values()))
    mse_iqr = np.subtract(*np.percentile(list(per_ab_mse.values()), [75, 25]))

    # Build DataFrames (MUST use flattened arrays)
    df = pd.DataFrame(X_test, columns=antibodies)
    df["Age_Group"] = age_groups
    df["Sex"] = sex

    df_recon = pd.DataFrame(reconstructed_flat, columns=antibodies)
    df_recon["Age_Group"] = age_groups
    df_recon["Sex"] = sex

    # MSE by age group
    mse_by_age = df.groupby("Age_Group").apply(
        lambda g: mean_squared_error(
            g[antibodies], df_recon.loc[g.index, antibodies]
        )
    ).to_dict()

    # MSE by sex
    mse_by_sex = df.groupby("Sex").apply(
        lambda g: mean_squared_error(
            g[antibodies], df_recon.loc[g.index, antibodies]
        )
    ).to_dict()

    return {
        "mse": mse,
        "mse_median": mse_median,
        "mse_iqr": mse_iqr,
        "per_antibody_mse": per_ab_mse,
        "mse_by_age_group": mse_by_age,
        "mse_by_sex": mse_by_sex,
        "N_test": len(X_test),
        "y_true": X_test,
        "y_pred": reconstructed_flat,
    }


In [ ]:
# ========== STEP 0: Set working directory (for running locally on laptop) =========
from __future__ import annotations
import sys, os, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("Repo:", REPO)

In [ ]:
import pickle

# Load mse_summaries from file
with open("mse_summaries.pkl", "rb") as f:
    mse_summaries = pickle.load(f)

# Confirm structure
print(mse_summaries.keys())


In [ ]:
# === Run this
study_list = ["SDY569", "SDY1625", "SDY524", "SDY797", "SDY1737"]
weights_dict = extract_model_weights_per_study(mse_summaries)

In [ ]:
# 1. Aggregate the weights
aggregated_weights = aggregate_weights_median(weights_dict)

# 2. Build federated model using any of the saved models as architecture reference
reference_model_path = "models/SDY569_model.keras"  # or any other
federated_model = build_federated_model(aggregated_weights, reference_model_path)



In [ ]:
federated_results = {}
for study in mse_summaries:
    print(f"Evaluating federated model on {study}...")
    federated_results[study] = evaluate_federated_model(federated_model, mse_summaries, study)


from pprint import pprint
pprint(federated_results[study_list[0]])

In [ ]:
comparison_data = []

for study in mse_summaries:
    local_mse = mse_summaries[study]["mse_median"]
    fed_mse = federated_results[study]["mse"]
    delta = local_mse - fed_mse
    pct_improvement = (delta / local_mse) * 100

    comparison_data.append({
        "Study": study,
        "Local MSE": round(local_mse, 4),
        "Federated MSE": round(fed_mse, 4),
        "Δ MSE": round(delta, 4),
        "% Improvement": f"{pct_improvement:.2f}%"
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)


## Retrain Federated Model on Each Study and Evaluate Stratified MSE

In [ ]:
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam
import numpy as np

# Store results after retraining
retrain_detailed_results = {}

for study in study_list:
    print(f"\n--- Retraining Federated Model on Study: {study} ---")

    summary = mse_summaries[study]
    
    # Extract X_train and reshape only relevant features
    X_train = summary["X_train"]
    antibodies = summary["antibody_labels"]

    # Number of features used in training (including one-hot encoded Age_Group and Sex)
    n_features_used = federated_model.input_shape[1]

    # Defensive check
    if X_train.shape[1] != n_features_used:
        print(f"  ⚠ Reshaping X_train: expected {n_features_used}, got {X_train.shape[1]}")
        # Try to truncate or slice appropriately
        X_train = X_train[:, :n_features_used]

    # Reshape for CNN input
    X_train_reshaped = X_train.reshape((-1, n_features_used, 1, 1))

    # Clone the federated model and copy weights
    local_model = clone_model(federated_model)
    local_model.set_weights(federated_model.get_weights())
    local_model.compile(optimizer=Adam(), loss='mse')

    # Retrain
    local_model.fit(X_train_reshaped, X_train_reshaped, epochs=10, batch_size=8, verbose=1)

    # Evaluate after retraining
    results_post = evaluate_federated_model(local_model, mse_summaries, study)
    retrain_detailed_results[study] = results_post


In [ ]:
retrain_detailed_results = {}

for study in study_list:
    summary = mse_summaries[study]
    X_train = summary["X_train"]
    n_features_used = federated_model.input_shape[1]
    
    if X_train.shape[1] != n_features_used:
        X_train = X_train[:, :n_features_used]
    
    X_train_reshaped = X_train.reshape((-1, n_features_used, 1, 1))
    
    # Clone & compile model
    local_model = clone_model(federated_model)
    local_model.set_weights(federated_model.get_weights())
    local_model.compile(optimizer=Adam(), loss='mse')
    
    # Retrain
    local_model.fit(X_train_reshaped, X_train_reshaped, epochs=10, batch_size=8, verbose=0)
    
    # Evaluate and save
    results_post = evaluate_federated_model(local_model, mse_summaries, study)
    retrain_detailed_results[study] = results_post


In [ ]:
from pprint import pprint
pprint(retrain_detailed_results[study_list[0]])


In [ ]:
pprint(mse_summaries[study_list[0]])


In [ ]:
import pandas as pd

rows = []

for study in study_list:
    try:
        fed_mse = mse_summaries[study]["mse_median"]
        retrain_mse = retrain_detailed_results[study]["mse_median"]

        rows.append({
            "Study": study,
            "Federated MSE (Median)": round(fed_mse, 6),
            "Retrained MSE (Median)": round(retrain_mse, 6),
            "Δ MSE": round(retrain_mse - fed_mse, 6),
            "% Change": round(100 * (retrain_mse - fed_mse) / fed_mse, 2)
        })
    except KeyError as e:
        print(f"Missing key {e} for study {study}")

df_summary = pd.DataFrame(rows)
df_summary


In [ ]:
import pandas as pd

rows = []

for study in study_list:
    local_mse = mse_summaries[study]["mse_median"]
    federated_mse = federated_results[study]["mse_median"]
    retrained_mse = retrain_detailed_results[study]["mse_median"]

    rows.append({
        "Study": study,
        "Local MSE": round(local_mse, 6),
        "Federated MSE": round(federated_mse, 6),
        "Retrained MSE": round(retrained_mse, 6),
        "Δ Fed - Local": round(federated_mse - local_mse, 6),
        "Δ Retrain - Local": round(retrained_mse - local_mse, 6),
    })

df_mse_comparison = pd.DataFrame(rows)
df_mse_comparison


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute % improvement columns
df_mse_comparison["% Improvement (Federated vs Local)"] = (
    (df_mse_comparison["Local MSE"] - df_mse_comparison["Federated MSE"]) / df_mse_comparison["Local MSE"]
) * 100

df_mse_comparison["% Improvement (Retrained vs Local)"] = (
    (df_mse_comparison["Local MSE"] - df_mse_comparison["Retrained MSE"]) / df_mse_comparison["Local MSE"]
) * 100

# Set up bar plot
x = np.arange(len(df_mse_comparison["Study"]))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2,
               df_mse_comparison["% Improvement (Federated vs Local)"],
               width, label="Federated vs Local")

bars2 = ax.bar(x + width/2,
               df_mse_comparison["% Improvement (Retrained vs Local)"],
               width, label="Retrained vs Local")

# Annotate bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', fontsize=8)

# Labels and formatting
ax.set_ylabel('% Improvement vs Local')
ax.set_title('Federated and Retrained Model Improvement vs Local (Median MSE)')
ax.set_xticks(x)
ax.set_xticklabels(df_mse_comparison["Study"])
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Save final MSE summary DataFrame to CSV
df_mse_comparison.to_csv("mse_comparison_summary.csv", index=False)


In [ ]:
mse_summaries[study]["per_antibody_mse"]


In [ ]:
study_list = ["SDY569", "SDY1625", "SDY524", "SDY797", "SDY1737"]


In [ ]:
import pandas as pd
import numpy as np



# === Per-Antibody MSE Table ===
antibody_rows = []

for study in study_list:
    local_ab = mse_summaries[study]["per_antibody_mse"]
    fed_ab = federated_results[study]["per_antibody_mse"]
    retrain_ab = retrain_detailed_results[study]["per_antibody_mse"]

    for ab in local_ab:
        local = local_ab.get(ab, np.nan)
        fed = fed_ab.get(ab, np.nan)
        retrained = retrain_ab.get(ab, np.nan)

        antibody_rows.append({
            "Study": study,
            "Antibody": ab,
            "Local MSE": local,
            "Federated MSE": fed,
            "Retrained MSE": retrained,
            "Δ Fed - Local": fed - local,
            "Δ Retrain - Local": retrained - local,
            "% Improvement (Fed vs Local)": ((local - fed) / local * 100) if local else np.nan,
            "% Improvement (Retrain vs Local)": ((local - retrained) / local * 100) if local else np.nan
        })

df_antibody_mse = pd.DataFrame(antibody_rows)

# === Sex-wise MSE (Federated) ===
sex_rows = []

for study in study_list:
    for sex, val in federated_results[study]["mse_by_sex"].items():
        sex_rows.append({
            "Study": study,
            "Sex": sex,
            "Federated MSE": val
        })

df_sex_mse = pd.DataFrame(sex_rows)

# === Age-group MSE (Federated) ===
age_rows = []

for study in study_list:
    for age, val in federated_results[study]["mse_by_age_group"].items():
        age_rows.append({
            "Study": study,
            "Age_Group": age,
            "Federated MSE": val
        })

df_age_mse = pd.DataFrame(age_rows)

# Display all
from IPython.display import display
display(df_antibody_mse)
display(df_sex_mse)
display(df_age_mse)


In [ ]:
import pandas as pd
import numpy as np

antibody_rows = []

for study in study_list:
    local_ab = mse_summaries[study]["per_antibody_mse"]
    fed_ab = federated_results[study]["per_antibody_mse"]
    retr_ab = retrain_detailed_results[study]["per_antibody_mse"]

    for ab in local_ab.keys():
        local = local_ab.get(ab, np.nan)
        fed = fed_ab.get(ab, np.nan)
        retr = retr_ab.get(ab, np.nan)

        antibody_rows.append({
            "Study": study,
            "Antibody": ab,
            "Local MSE (median)": local,
            "Federated MSE (median)": fed,
            "Retrained MSE (median)": retr,
            "Δ Fed − Local": fed - local,
            "Δ Retrain − Local": retr - local,
            "% Improvement Fed vs Local": ((local - fed) / local * 100) if local > 0 else np.nan,
            "% Improvement Retrain vs Local": ((local - retr) / local * 100) if local > 0 else np.nan,
        })

df_antibody_mse = pd.DataFrame(antibody_rows)


In [ ]:
from sklearn.metrics import mean_squared_error

sex_rows = []

for study in study_list:
    summary = mse_summaries[study]

    X_true = summary["X_test"]
    X_recon_local = summary["reconstructed"]
    sex_labels = summary["Sex"]

    for sex in np.unique(sex_labels):
        idx = sex_labels == sex

        # Local (median of squared error)
        local_mse = np.median((X_true[idx] - X_recon_local[idx]) ** 2)

        # Federated (already computed)
        fed_mse = federated_results[study]["mse_by_sex"].get(sex, np.nan)

        # Retrained (already computed)
        retr_mse = retrain_detailed_results[study]["mse_by_sex"].get(sex, np.nan)

        sex_rows.append({
            "Study": study,
            "Sex": sex,
            "Local MSE (median)": local_mse,
            "Federated MSE (median)": fed_mse,
            "Retrained MSE (median)": retr_mse,
            "Δ Fed − Local": fed_mse - local_mse,
            "Δ Retrain − Local": retr_mse - local_mse,
        })

df_sex_mse = pd.DataFrame(sex_rows)


In [ ]:
age_rows = []

for study in study_list:
    summary = mse_summaries[study]

    X_true = summary["X_test"]
    X_recon_local = summary["reconstructed"]
    age_labels = summary["Age_Group"]

    for age in np.unique(age_labels):
        idx = age_labels == age

        # Local
        local_mse = np.median((X_true[idx] - X_recon_local[idx]) ** 2)

        # Federated
        fed_mse = federated_results[study]["mse_by_age_group"].get(age, np.nan)

        # Retrained
        retr_mse = retrain_detailed_results[study]["mse_by_age_group"].get(age, np.nan)

        age_rows.append({
            "Study": study,
            "Age_Group": age,
            "Local MSE (median)": local_mse,
            "Federated MSE (median)": fed_mse,
            "Retrained MSE (median)": retr_mse,
            "Δ Fed − Local": fed_mse - local_mse,
            "Δ Retrain − Local": retr_mse - local_mse,
        })

df_age_mse = pd.DataFrame(age_rows)


In [ ]:
from IPython.display import display

display(df_antibody_mse)
display(df_sex_mse)
display(df_age_mse)

df_antibody_mse.to_csv("per_antibody_mse_median.csv", index=False)
df_sex_mse.to_csv("per_sex_mse_median.csv", index=False)
df_age_mse.to_csv("per_age_group_mse_median.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure % improvement columns are numeric
df_antibody_mse["% Improvement (Fed vs Local)"] = pd.to_numeric(
    df_antibody_mse["% Improvement (Fed vs Local)"], errors="coerce"
)
df_antibody_mse["% Improvement (Retrain vs Local)"] = pd.to_numeric(
    df_antibody_mse["% Improvement (Retrain vs Local)"], errors="coerce"
)

# Plot: Per-Antibody % Improvement (Federated vs Local)
plt.figure(figsize=(14, 6))
sns.barplot(
    data=df_antibody_mse,
    x="Antibody",
    y="% Improvement (Fed vs Local)",
    hue="Study"
)
plt.axhline(0, color="gray", linestyle="--")
plt.title("Per-Antibody % Improvement (Federated vs Local)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Plot: Per-Antibody % Improvement (Retrain vs Local)
plt.figure(figsize=(14, 6))
sns.barplot(
    data=df_antibody_mse,
    x="Antibody",
    y="% Improvement (Retrain vs Local)",
    hue="Study"
)
plt.axhline(0, color="gray", linestyle="--")
plt.title("Per-Antibody % Improvement (Retrained vs Local)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
pprint(df_antibody_mse)
pprint(df_sex_mse)
pprint(df_age_mse)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure improvement columns are numeric
df_antibody_mse["% Improvement Fed vs Local"] = pd.to_numeric(df_antibody_mse["% Improvement Fed vs Local"], errors="coerce")
df_antibody_mse["% Improvement Retrain vs Local"] = pd.to_numeric(df_antibody_mse["% Improvement Retrain vs Local"], errors="coerce")

# --- Plot 1: Per-Antibody % Improvement (Federated vs Local) ---
plt.figure(figsize=(16, 6))
sns.barplot(data=df_antibody_mse, x="Antibody", y="% Improvement Fed vs Local", hue="Study")
plt.axhline(0, color="gray", linestyle="--")
plt.title("Per-Antibody % Improvement (Federated vs Local)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Plot 2: Per-Antibody % Improvement (Retrained vs Local) ---
plt.figure(figsize=(16, 6))
sns.barplot(data=df_antibody_mse, x="Antibody", y="% Improvement Retrain vs Local", hue="Study")
plt.axhline(0, color="gray", linestyle="--")
plt.title("Per-Antibody % Improvement (Retrained vs Local)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Plot 3: Sex-wise MSE Comparison ---
melted_sex = df_sex_mse.melt(id_vars=["Study", "Sex"], 
                             value_vars=["Local MSE (median)", "Federated MSE (median)", "Retrained MSE (median)"],
                             var_name="Model", value_name="MSE")

plt.figure(figsize=(12, 6))
sns.barplot(data=melted_sex, x="Sex", y="MSE", hue="Model")
plt.title("Sex-wise Median MSE by Model")
plt.tight_layout()
plt.show()

# --- Plot 4: Age-group MSE Comparison ---
melted_age = df_age_mse.melt(id_vars=["Study", "Age_Group"], 
                             value_vars=["Local MSE (median)", "Federated MSE (median)", "Retrained MSE (median)"],
                             var_name="Model", value_name="MSE")

plt.figure(figsize=(12, 6))
sns.barplot(data=melted_age, x="Age_Group", y="MSE", hue="Model")
plt.title("Age Group-wise Median MSE by Model")
plt.tight_layout()
plt.show()

